# Import Library

In [1]:
# ==========================================
# CELL 1: IMPORT & INISIALISASI
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from tqdm import tqdm

# NLP Augmentation
import nlpaug.augmenter.word as naw

# Sastrawi untuk TF-IDF
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# DagsHub & MLflow
import dagshub
import mlflow
import mlflow.sklearn

# 1. Cek GPU (CUDA)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Menggunakan perangkat: {device}")

# 2. Inisialisasi Sastrawi
print("⏳ Menyiapkan Sastrawi Stemmer & Stopword...")
stemmer = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

print("✅ Cell 1 Selesai: Semua library siap tempur!")

d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Menggunakan perangkat: cuda
⏳ Menyiapkan Sastrawi Stemmer & Stopword...
✅ Cell 1 Selesai: Semua library siap tempur!


# Import Data

In [2]:
# ==========================================
# CELL 2: LOAD DATA & AUGMENTASI BERBASIS INDOBERT
# ==========================================
file_path = "../../data/dataset_keluhan_ntg.csv" # Sesuaikan path

# 1. Load Data
df = pd.read_csv(file_path, sep='|', on_bad_lines='skip')
kolom_target = ['kategori_aset', 'severity', 'root_cause', 'jenis_kerusakan']
df = df.dropna(subset=kolom_target + ['teks_keluhan_awam', 'teks_laporan_teknisi'])

# 2. Gabungkan Teks
df['input_teks'] = df['teks_keluhan_awam'].astype(str) + " " + df['teks_laporan_teknisi'].astype(str)

Y = df[kolom_target]
X_raw = df['input_teks']

# Basic Clean (Hapus simbol saja, biarkan kata utuh untuk Augmentasi)
def basic_clean(text):
    text = str(text).lower()
    return re.sub(r'[^a-z0-9\s]', ' ', text).strip()

X_clean_basic = X_raw.apply(basic_clean)

# 3. Augmentasi Menggunakan IndoBERT (Sama persis dengan script sebelumnya)
print("⏳ Memulai Augmentasi Data (Menggandakan variasi kalimat di GPU)...")
aug = naw.ContextualWordEmbsAug(
    model_path='indobenchmark/indobert-base-p1', 
    action="substitute", 
    device='cuda' if torch.cuda.is_available() else 'cpu',
    aug_p=0.2 
)

X_aug_list = []
Y_aug_list = []

for i, teks in enumerate(tqdm(X_clean_basic, desc="Augmentasi")):
    # Simpan teks asli
    X_aug_list.append(teks)
    Y_aug_list.append(Y.iloc[i].values)
    
    # Buat variasi teks
    teks_baru = aug.augment(teks)
    if isinstance(teks_baru, list):
        teks_baru = teks_baru[0]
        
    X_aug_list.append(teks_baru)
    Y_aug_list.append(Y.iloc[i].values)

X_augmented = pd.Series(X_aug_list)
Y_final = pd.DataFrame(Y_aug_list, columns=kolom_target)

print(f"✅ Augmentasi Selesai! Data melonjak dari {len(X_clean_basic)} menjadi {len(X_augmented)} baris.")

⏳ Memulai Augmentasi Data (Menggandakan variasi kalimat di GPU)...


d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\transformers\modeling_utils.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer b

✅ Augmentasi Selesai! Data melonjak dari 415 menjadi 830 baris.


# Cleaning & Preprocessing

In [3]:
# ==========================================
# CELL 3: SASTRAWI PREPROCESSING (STEMMING & STOPWORDS)
# ==========================================
def sastrawi_clean(text):
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', str(text)).strip() 
    # Hapus kata hubung (di, ke, dari, yang)
    text = stopword.remove(text) 
    # Kembalikan ke kata dasar (memperbaiki -> baik)
    text = stemmer.stem(text) 
    return text

print("⏳ Memulai proses Sastrawi pada data Augmented (Ini memakan waktu sekitar 1-2 menit)...")

# Gunakan tqdm di apply untuk melihat progress
tqdm.pandas(desc="Sastrawi Processing")
X_final_sastrawi = X_augmented.progress_apply(sastrawi_clean)

print("\n✅ Cell 3 Selesai: Teks sudah berbentuk kata dasar murni dan siap di-Vektorisasi!")

⏳ Memulai proses Sastrawi pada data Augmented (Ini memakan waktu sekitar 1-2 menit)...


Sastrawi Processing: 100%|██████████| 830/830 [01:49<00:00,  7.56it/s]


✅ Cell 3 Selesai: Teks sudah berbentuk kata dasar murni dan siap di-Vektorisasi!


In [4]:
# ==========================================
# CELL 4: TRAINING & MLFLOW TRACKING (TF-IDF)
# ==========================================
# 1. Split Data
X_train, X_test, y_train, y_test = train_test_split(X_final_sastrawi, Y_final, test_size=0.2, random_state=42)

# 2. Inisialisasi DagsHub
dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)

with mlflow.start_run(run_name="Eksperimen_4_TFIDF_Augmented"):
    
    # 3. Bangun Pipeline (TF-IDF langsung nyambung ke Random Forest)
    # n-gram (1,2) berarti membaca per 1 kata dan per 2 kata berpasangan (misal: "ac mati")
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(1, 2))),
        ('clf', MultiOutputClassifier(RandomForestClassifier(n_estimators=200, random_state=42)))
    ])
    
    # 4. Training
    print("🤖 Mulai melatih model TF-IDF + Random Forest...")
    pipeline.fit(X_train, y_train)
    
    # 5. Prediksi & Evaluasi
    y_pred = pipeline.predict(X_test)
    
    # Exact Match Ratio
    exact_match = np.all(y_pred == y_test.values, axis=1).mean()
    print(f"\n🎯 Exact Match Ratio (Akurasi Penuh): {exact_match:.2%}")
    mlflow.log_metric("exact_match_ratio", exact_match)
    
    # Akurasi Per Kolom
    print("-" * 30)
    for i, col in enumerate(kolom_target):
        acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
        print(f"🔸 Akurasi {col.upper()}: {acc:.2%}")
        mlflow.log_metric(f"accuracy_{col}", acc)
        
    # 6. Catat Parameter & Simpan
    mlflow.log_param("Feature_Extractor", "TF-IDF (Sastrawi)")
    mlflow.log_param("Augmentation", "Contextual Word Embedding (nlpaug)")
    mlflow.log_param("Model", "Random Forest")
    mlflow.log_param("max_features", 2000)
    
    # Simpan model utuh
    mlflow.sklearn.log_model(pipeline, "model_tfidf_rf")
    
    print("\n✅ Cell 4 Selesai: Hasil Eksperimen TF-IDF berhasil dikirim ke DagsHub!")

Accessing as NazeeraAlthea

Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!

🤖 Mulai melatih model TF-IDF + Random Forest...

🎯 Exact Match Ratio (Akurasi Penuh): 84.94%
------------------------------
🔸 Akurasi KATEGORI_ASET: 96.99%
🔸 Akurasi SEVERITY: 90.96%
🔸 Akurasi ROOT_CAUSE: 96.39%
🔸 Akurasi JENIS_KERUSAKAN: 97.59%


2026/05/15 16:40:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 16:40:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Cell 4 Selesai: Hasil Eksperimen TF-IDF berhasil dikirim ke DagsHub!
🏃 View run Eksperimen_4_TFIDF_Augmented at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0/runs/d79982530e52463bbe4648583a8c36d5
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0


In [5]:
# ==========================================
# CELL 5: BLIND TESTING (UJI NYALI MODEL)
# ==========================================

print("=== 🧪 MEMULAI BLIND TESTING (OUT-OF-DISTRIBUTION) ===\n")

# 1. Buat data keluhan acak ala HRD yang panik (Tidak pernah ada di dataset)
keluhan_baru = [
    # Tes 1: Bahasa gaul + Typo + Panik (Target Logis: AC, Sedang/Berat, Bocor)
    "Tolonggg dong ini ac di ruang rapat lt 3 netes airnya parah bgt ngerusak karpet!! cepet di benerin ya mas",
    
    # Tes 2: Gejala aneh + Kategori Meleset (Target Logis: Lift, Fatal, Sistem error)
    "Waduh lift penumpang lantai 5 tiba tiba nyangkut di tengah jalan trus lampunya mati. ada orang di dalem woy!",
    
    # Tes 3: Laporan teknis banget tapi beda gaya nulis (Target Logis: Granit/Civil, Ringan, Aus normal)
    "Lantai granit depan lobby utama kelihatan kusam dan warnanya agak pudar, kayaknya perlu dipoles ulang deh biar kinclong lagi."
]

# 2. Karena TF-IDF kita menggunakan Sastrawi, kita WAJIB me-run fungsi Sastrawi 
#    yang sama persis ke keluhan baru ini sebelum dimasukkan ke model.
print("⏳ Sedang memproses teks baru menggunakan Sastrawi...")

# (Pastikan fungsi sastrawi_clean dari Cell 3 masih ada di memori)
keluhan_clean = [sastrawi_clean(teks) for teks in keluhan_baru]

# 3. Lempar ke Pipeline (Pipeline akan otomatis melakukan TF-IDF lalu Random Forest)
# Perhatikan: Kita pakai variabel 'pipeline' dari hasil training di Cell 4
print("🤖 Meminta Model Random Forest untuk memprediksi...\n")
hasil_prediksi = pipeline.predict(keluhan_clean)

# 4. Tampilkan Hasil dengan Cantik
for i, teks_asli in enumerate(keluhan_baru):
    print(f"📝 KELUHAN AWAM #{i+1}:")
    print(f"\"{teks_asli}\"")
    print("-" * 30)
    print("🎯 TEBAKAN AI (MULTIPLE TARGET):")
    
    # Hasil prediksi bentuknya array 2D. Kolom target kita:
    # ['kategori_aset', 'severity', 'root_cause', 'jenis_kerusakan']
    print(f" 🔹 Kategori      : {hasil_prediksi[i][0]}")
    print(f" 🔹 Severity      : {hasil_prediksi[i][1]}")
    print(f" 🔹 Root Cause    : {hasil_prediksi[i][2]}")
    print(f" 🔹 Kerusakan     : {hasil_prediksi[i][3]}")
    print("=" * 50 + "\n")

=== 🧪 MEMULAI BLIND TESTING (OUT-OF-DISTRIBUTION) ===

⏳ Sedang memproses teks baru menggunakan Sastrawi...
🤖 Meminta Model Random Forest untuk memprediksi...

📝 KELUHAN AWAM #1:
"Tolonggg dong ini ac di ruang rapat lt 3 netes airnya parah bgt ngerusak karpet!! cepet di benerin ya mas"
------------------------------
🎯 TEBAKAN AI (MULTIPLE TARGET):
 🔹 Kategori      : Mechanical
 🔹 Severity      : Berat
 🔹 Root Cause    : Aus normal
 🔹 Kerusakan     : Korsleting

📝 KELUHAN AWAM #2:
"Waduh lift penumpang lantai 5 tiba tiba nyangkut di tengah jalan trus lampunya mati. ada orang di dalem woy!"
------------------------------
🎯 TEBAKAN AI (MULTIPLE TARGET):
 🔹 Kategori      : Electrical
 🔹 Severity      : Ringan
 🔹 Root Cause    : Overload
 🔹 Kerusakan     : Lampu mati

📝 KELUHAN AWAM #3:
"Lantai granit depan lobby utama kelihatan kusam dan warnanya agak pudar, kayaknya perlu dipoles ulang deh biar kinclong lagi."
------------------------------
🎯 TEBAKAN AI (MULTIPLE TARGET):
 🔹 Kategori     